In [1]:
!pip install ytmusicapi

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import ytmusicapi
from ytmusicapi import YTMusic

In [ ]:
from datetime import datetime
import json
import csv

from ytmusicapi import YTMusic
import pandas as pd
import time

with open("client_secret_tv.json", "r", encoding="utf-8") as f:
    client_secret = json.load(f)["installed"]

client_id = client_secret["client_id"]
client_secret = client_secret["client_secret"]

oauth_credentials = {
    "client_id": client_id,
    "client_secret": client_secret,
}

# OAuth認証ファイルを使用して初期化
yt = YTMusic("browser.json")

def fetch_filtered_songs(query, target_count=200, filename_prefix="yt_music_export"):
    exclude_keywords = ["mix", "cumbia", "mosaic", "medley", "live", "cover", "remix", "karaoke"]
    results_list = []
    fetch_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"『{query}』で検索中...")

    # filter="songs"を追加して曲のみ検索し、limitを500に増やしてフィルタリング後もtarget_count確保しやすくする
    raw_results = yt.search(query, filter="songs", limit=500)
    print(f"raw_results count: {len(raw_results)}")  # デバッグ用

    for item in raw_results:
        if len(results_list) >= target_count:
            break

        title = item.get('title', '').strip()
        artists = ", ".join([a.get('name','').strip() for a in item.get('artists', []) if a.get('name')])

        # titleやartistsが空の場合、スキップ（データが取れていない場合の対策）
        if not title or not artists:
            print(f"スキップ: データ不足 - {item}")
            continue

        # キーワード除外判定（titleとartistsの両方をチェック）
        combined_text = (title + artists).lower()
        if any(word in combined_text for word in exclude_keywords):
            continue

        # データの格納 (album が None の場合に対応)
        album_name = (item.get('album') or {}).get('name', 'N/A')
        video_id = item.get('videoId') or ''
        url = f"https://www.youtube.com/watch?v={video_id}" if video_id else ''

        # 説明の取得（可能なら取得、失敗しても空文字にする）
        description = ''
        if video_id:
            try:
                # API呼び出しが多くならないよう短時間待つ（レート制限対策）
                time.sleep(0.2)
                song_details = yt.get_song(video_id)
                # ytmusicapi の戻り値の構造を考慮して短縮説明・フル説明を探す
                description = (
                    song_details.get('videoDetails', {}).get('shortDescription', '')
                    or song_details.get('microformat', {}).get('playerMicroformatRenderer', {}).get('description', '')
                    or ''
                )
            except Exception as e:
                print(f"説明取得エラー for {video_id}: {e}")
                description = ''

        results_list.append({
            "title": title,
            "artist": artists,
            "album": album_name,
            "url": url,
            "description": description,
            "release_year": item.get('year', 'N/A'),
            "fetch_date": fetch_date
        })

        # print時にアーティスト名、タイトル、URLを出力
        print(f"[{len(results_list)}] 取得: {artists} - {title} - {url}")

    # 結果がtarget_countに達しなかった場合の警告
    if len(results_list) < target_count:
        print(f"警告: 目標の{target_count}曲に達しませんでした。取得数: {len(results_list)}")

    # 保存（Excel対応のエンコーディング）
    if results_list:
        filename = f"{filename_prefix}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
        with open(filename, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=results_list[0].keys())
            writer.writeheader()
            writer.writerows(results_list)

        print(f"\n完了！ '{filename}' に保存しました。")
    else:
        print("結果がありませんでした。")


In [4]:
# 実行例
fetch_filtered_songs("ecuador folklore sanjuanito", target_count=200, filename_prefix="sanjuanito")
fetch_filtered_songs("bolivia folklore tobas", target_count=200, filename_prefix="tobas")

『ecuador folklore sanjuanito』で検索中...
raw_results count: 492
[1] 取得: Proyección - Ñucallajta - https://www.youtube.com/watch?v=SeMeKsRY7WU
[2] 取得: Ecuador Manta - Ecuador San Juanito - https://www.youtube.com/watch?v=IXVUwOqt5XE
[3] 取得: Ecuador Causary - Naupa Llacta - https://www.youtube.com/watch?v=Y75gLE0ihyA
[4] 取得: Ñanda Mañachi - Ñuca Llacta - https://www.youtube.com/watch?v=xRei8rigCOA
[5] 取得: Proyección - Selección de San Juanitos - https://www.youtube.com/watch?v=9nuMFtP71jY
[6] 取得: Fusión Inka - Tabacundeña (Sanjuanito) - https://www.youtube.com/watch?v=1hLCpigJ024
[7] 取得: Ecuador Causary - Yana Cuchi - https://www.youtube.com/watch?v=C28-TX26Zvk
[8] 取得: Kamari Ec - Rosalia - https://www.youtube.com/watch?v=8KOpAa_hnG8
[9] 取得: Ecuador Causary - Carmelita - https://www.youtube.com/watch?v=6493VpdphCE
[10] 取得: Ecuador Causary - Mashicuna - https://www.youtube.com/watch?v=gQy3yD6N86s
[11] 取得: Ecuador Manta - Cuatro Esquinas - https://www.youtube.com/watch?v=DPS-SIGWyh8
[12] 取得: E